# Machine Learning Modeling — Healthcare Disease Prediction

> **Prerequisite:** Run `01_EDA.ipynb` → `02_Data_Preprocessing.ipynb` first.

## Objectives
1. Establish baseline model performance across 9 classifiers.
2. Select best candidates via StratifiedKFold cross-validation.
3. Tune hyperparameters with RandomizedSearchCV (time-efficient).
4. Evaluate with healthcare-appropriate metrics: Recall, F1, ROC-AUC, MCC.
5. Produce publication-quality plots: ROC curves, PR curves, confusion matrices.
6. Export the best model per dataset as `.joblib`.

In [ ]:
# ── 0. Imports & Config ────────────────────────────────────────────────────
import time, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model     import LogisticRegression
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import (
    RandomForestClassifier, ExtraTreesClassifier,
    AdaBoostClassifier, GradientBoostingClassifier
)
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection  import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics          import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, balanced_accuracy_score, matthews_corrcoef,
    confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, PrecisionRecallDisplay, classification_report,
    average_precision_score
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
CV_FOLDS = 5
N_ITER   = 20  # RandomizedSearchCV iterations
os.makedirs('plots', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
print('✅ Imports complete')

In [ ]:
# ── 1. Load Preprocessed Data ──────────────────────────────────────────────

def load_splits(prefix):
    X_tr = pd.read_csv(f'{prefix}_X_train.csv').values
    X_te = pd.read_csv(f'{prefix}_X_test.csv').values
    y_tr = pd.read_csv(f'{prefix}_y_train.csv').values.ravel().astype(int)
    y_te = pd.read_csv(f'{prefix}_y_test.csv').values.ravel().astype(int)
    print(f'{prefix:10s}: X_train={X_tr.shape}, X_test={X_te.shape}')
    return X_tr, X_te, y_tr, y_te

X_tr_c, X_te_c, y_tr_c, y_te_c = load_splits('cardio')
X_tr_d, X_te_d, y_tr_d, y_te_d = load_splits('diabetes')
X_tr_h, X_te_h, y_tr_h, y_te_h = load_splits('heart')

datasets = {
    'Cardiovascular': (X_tr_c, X_te_c, y_tr_c, y_te_c),
    'Diabetes':       (X_tr_d, X_te_d, y_tr_d, y_te_d),
    'Heart Disease':  (X_tr_h, X_te_h, y_tr_h, y_te_h),
}
print('✅ Data loaded')

## Baseline Models

We train 9 classifiers with default hyperparameters and rank them by cross-validated Recall (primary metric for disease detection — missing a positive case is more costly than a false alarm).

In [ ]:
# ── 2. Baseline Model Definitions ─────────────────────────────────────────

BASELINE_MODELS = {
    'Logistic Regression':   LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree':         DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest':         RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE),
    'Extra Trees':           ExtraTreesClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE),
    'XGBoost':               XGBClassifier(n_estimators=100, use_label_encoder=False,
                                           eval_metric='logloss', n_jobs=-1, random_state=RANDOM_STATE),
    'LightGBM':              LGBMClassifier(n_estimators=100, n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
    'CatBoost':              CatBoostClassifier(iterations=100, verbose=0, random_state=RANDOM_STATE),
    'AdaBoost':              AdaBoostClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Gradient Boosting':     GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
}
print(f'✅ {len(BASELINE_MODELS)} baseline models defined')

In [ ]:
# ── 3. Cross-Validation Baseline Evaluation ────────────────────────────────

def run_baseline(X_train, y_train, dataset_name):
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    results = []
    print(f'\n⏳ Baseline CV — {dataset_name}')
    for name, model in BASELINE_MODELS.items():
        t0 = time.time()
        recall   = cross_val_score(model, X_train, y_train, cv=skf, scoring='recall',   n_jobs=-1).mean()
        f1       = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1',       n_jobs=-1).mean()
        roc_auc  = cross_val_score(model, X_train, y_train, cv=skf, scoring='roc_auc',  n_jobs=-1).mean()
        accuracy = cross_val_score(model, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1).mean()
        elapsed  = time.time() - t0
        results.append({'Model': name, 'Recall': recall, 'F1': f1,
                        'ROC-AUC': roc_auc, 'Accuracy': accuracy, 'Time(s)': elapsed})
        print(f'   {name:25s}  Recall={recall:.3f}  F1={f1:.3f}  AUC={roc_auc:.3f}  [{elapsed:.1f}s]')

    df_res = pd.DataFrame(results).sort_values('Recall', ascending=False).reset_index(drop=True)
    return df_res

baseline_cardio = run_baseline(X_tr_c, y_tr_c, 'Cardiovascular')
baseline_diab   = run_baseline(X_tr_d, y_tr_d, 'Diabetes')
baseline_heart  = run_baseline(X_tr_h, y_tr_h, 'Heart Disease')

In [ ]:
# ── 4. Baseline Comparison Plots ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for ax, (df_res, title) in zip(axes, [
    (baseline_cardio, 'Cardiovascular'),
    (baseline_diab,   'Diabetes'),
    (baseline_heart,  'Heart Disease'),
]):
    x = np.arange(len(df_res))
    width = 0.22
    ax.bar(x - width*1.5, df_res['Recall'],   width, label='Recall',   color='#E53935', alpha=0.85)
    ax.bar(x - width*0.5, df_res['F1'],       width, label='F1',       color='#1976D2', alpha=0.85)
    ax.bar(x + width*0.5, df_res['ROC-AUC'],  width, label='ROC-AUC',  color='#388E3C', alpha=0.85)
    ax.bar(x + width*1.5, df_res['Accuracy'], width, label='Accuracy', color='#F57C00', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(df_res['Model'], rotation=40, ha='right', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_title(f'{title} — Baseline CV Metrics', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('plots/baseline_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Saved: plots/baseline_comparison.png')

## Hyperparameter Optimization

We select the **top-2 models by Recall** from baseline and tune them with `RandomizedSearchCV` (n_iter=20, cv=5). This balances thoroughness with time constraints.

In [ ]:
# ── 5. Hyperparameter Search Spaces ───────────────────────────────────────

SEARCH_SPACES = {
    'Random Forest': {
        'model': RandomForestClassifier(n_jobs=-1, random_state=RANDOM_STATE),
        'params': {
            'n_estimators':      [100, 200, 300],
            'max_depth':         [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf':  [1, 2, 4],
            'class_weight':      ['balanced', None],
        }
    },
    'Extra Trees': {
        'model': ExtraTreesClassifier(n_jobs=-1, random_state=RANDOM_STATE),
        'params': {
            'n_estimators':      [100, 200, 300],
            'max_depth':         [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'class_weight':      ['balanced', None],
        }
    },
    'XGBoost': {
        'model': XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                               n_jobs=-1, random_state=RANDOM_STATE),
        'params': {
            'n_estimators':  [100, 200, 300],
            'max_depth':     [3, 5, 7, 9],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'subsample':     [0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
            'scale_pos_weight': [1, 3, 5, 10],  # handles imbalance
        }
    },
    'LightGBM': {
        'model': LGBMClassifier(n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
        'params': {
            'n_estimators':   [100, 200, 300],
            'max_depth':      [-1, 10, 20],
            'learning_rate':  [0.01, 0.05, 0.1, 0.2],
            'num_leaves':     [31, 63, 127],
            'class_weight':   ['balanced', None],
        }
    },
    'CatBoost': {
        'model': CatBoostClassifier(verbose=0, random_state=RANDOM_STATE),
        'params': {
            'iterations':   [100, 200, 300],
            'depth':        [4, 6, 8, 10],
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'l2_leaf_reg':  [1, 3, 5, 7],
        }
    },
}
print('✅ Hyperparameter search spaces defined')

In [ ]:
# ── 6. Tuning Function ─────────────────────────────────────────────────────

def tune_model(model_name, X_train, y_train, dataset_name):
    if model_name not in SEARCH_SPACES:
        print(f'  ⚠️ No search space for {model_name}, using baseline')
        return BASELINE_MODELS[model_name]

    config = SEARCH_SPACES[model_name]
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    t0 = time.time()

    search = RandomizedSearchCV(
        config['model'],
        config['params'],
        n_iter=N_ITER,
        scoring='recall',     # primary metric for healthcare
        cv=skf,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        refit=True,
        verbose=0
    )
    search.fit(X_train, y_train)
    elapsed = time.time() - t0
    print(f'  ✅ {dataset_name} | {model_name}: Best Recall={search.best_score_:.3f} [{elapsed:.1f}s]')
    print(f'     Best params: {search.best_params_}')
    return search.best_estimator_

In [ ]:
# ── 7. Select Top-2 and Tune Per Dataset ──────────────────────────────────

def get_top_models(baseline_df, n=2):
    return baseline_df.head(n)['Model'].tolist()

tuned_models = {}

for ds_name, (baseline_df, X_tr, y_tr) in [
    ('Cardiovascular', (baseline_cardio, X_tr_c, y_tr_c)),
    ('Diabetes',       (baseline_diab,   X_tr_d, y_tr_d)),
    ('Heart Disease',  (baseline_heart,  X_tr_h, y_tr_h)),
]:
    top_names = get_top_models(baseline_df)
    print(f'\n🔧 Tuning for {ds_name}: {top_names}')
    tuned_models[ds_name] = {}
    for m_name in top_names:
        tuned_models[ds_name][m_name] = tune_model(m_name, X_tr, y_tr, ds_name)

print('\n✅ Hyperparameter tuning complete')

## Performance Evaluation

In [ ]:
# ── 8. Comprehensive Evaluation Function ──────────────────────────────────

def evaluate_model(model, X_test, y_test, model_name, dataset_name):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        'Dataset':             dataset_name,
        'Model':               model_name,
        'Accuracy':            accuracy_score(y_test, y_pred),
        'Precision':           precision_score(y_test, y_pred, zero_division=0),
        'Recall':              recall_score(y_test, y_pred),
        'F1':                  f1_score(y_test, y_pred),
        'ROC-AUC':             roc_auc_score(y_test, y_proba),
        'PR-AUC':              average_precision_score(y_test, y_proba),
        'Balanced Accuracy':   balanced_accuracy_score(y_test, y_pred),
        'MCC':                 matthews_corrcoef(y_test, y_pred),
    }
    return metrics, y_pred, y_proba


all_metrics = []
test_results = {}  # store predictions for plotting

for ds_name, (X_tr, X_te, y_tr, y_te) in [
    ('Cardiovascular', (X_tr_c, X_te_c, y_tr_c, y_te_c)),
    ('Diabetes',       (X_tr_d, X_te_d, y_tr_d, y_te_d)),
    ('Heart Disease',  (X_tr_h, X_te_h, y_tr_h, y_te_h)),
]:
    test_results[ds_name] = {'y_test': y_te, 'models': {}}
    for m_name, model in tuned_models[ds_name].items():
        metrics, y_pred, y_proba = evaluate_model(model, X_te, y_te, m_name, ds_name)
        all_metrics.append(metrics)
        test_results[ds_name]['models'][m_name] = {'model': model, 'y_pred': y_pred, 'y_proba': y_proba}

metrics_df = pd.DataFrame(all_metrics)
print('\n📊 Test Set Evaluation Results:')
display(metrics_df.round(4))

In [ ]:
# ── 9. ROC Curves ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, ds_name in zip(axes, ['Cardiovascular', 'Diabetes', 'Heart Disease']):
    y_test = test_results[ds_name]['y_test']
    for m_name, res in test_results[ds_name]['models'].items():
        RocCurveDisplay.from_predictions(y_test, res['y_proba'], name=m_name, ax=ax)
    ax.plot([0,1],[0,1],'k--', alpha=0.4)
    ax.set_title(f'{ds_name} — ROC Curves', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/roc_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 10. Precision-Recall Curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, ds_name in zip(axes, ['Cardiovascular', 'Diabetes', 'Heart Disease']):
    y_test = test_results[ds_name]['y_test']
    for m_name, res in test_results[ds_name]['models'].items():
        PrecisionRecallDisplay.from_predictions(y_test, res['y_proba'], name=m_name, ax=ax)
    baseline_pr = y_test.mean()
    ax.axhline(baseline_pr, color='k', linestyle='--', alpha=0.4, label=f'Random ({baseline_pr:.2f})')
    ax.set_title(f'{ds_name} — Precision-Recall Curves', fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('plots/pr_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 11. Confusion Matrices (best model per dataset) ────────────────────────

def select_best_model(ds_name):
    """Select model with best Recall from tuned candidates."""
    best_m, best_r = None, -1
    for m_name, res in test_results[ds_name]['models'].items():
        r = recall_score(test_results[ds_name]['y_test'], res['y_pred'])
        if r > best_r:
            best_r, best_m = r, m_name
    return best_m

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
best_models_per_ds = {}

for ax, ds_name in zip(axes, ['Cardiovascular', 'Diabetes', 'Heart Disease']):
    best_name = select_best_model(ds_name)
    best_models_per_ds[ds_name] = best_name
    res = test_results[ds_name]['models'][best_name]
    y_test = test_results[ds_name]['y_test']
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Disease', 'Disease'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{ds_name}\nBest: {best_name}', fontweight='bold')

plt.suptitle('Confusion Matrices — Best Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n🏆 Best model per dataset (by Recall):')
for ds, m in best_models_per_ds.items():
    print(f'   {ds}: {m}')

## Feature Importance

In [ ]:
# ── 12. Feature Importance Plots ──────────────────────────────────────────

# Feature name lists (must match ColumnTransformer output order)
cardio_feature_names = ['age_years','height','weight','ap_hi','ap_lo','bmi','pulse_pressure',
                         'gender','cholesterol','gluc','smoke','alco','active','hypertension','overweight']
brfss_feature_names  = ['BMI','MentHlth','PhysHlth','lifestyle_score','comorbidity_count','health_burden',
                         'HighBP','HighChol','CholCheck','Smoker','Stroke',
                         'PhysActivity','Fruits','Veggies','HvyAlcoholConsump',
                         'AnyHealthcare','NoDocbcCost','DiffWalk','Sex',
                         'GenHlth','Age','Education','Income']

feature_names_map = {
    'Cardiovascular': cardio_feature_names,
    'Diabetes':       brfss_feature_names,
    'Heart Disease':  brfss_feature_names,
}

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

for ax, ds_name in zip(axes, ['Cardiovascular', 'Diabetes', 'Heart Disease']):
    best_name = best_models_per_ds[ds_name]
    model = tuned_models[ds_name][best_name]
    feature_names = feature_names_map[ds_name]

    if hasattr(model, 'feature_importances_'):
        imp = model.feature_importances_
        n = min(len(imp), len(feature_names))
        imp_series = pd.Series(imp[:n], index=feature_names[:n]).sort_values(ascending=True).tail(15)
        imp_series.plot(kind='barh', ax=ax, color=sns.color_palette('viridis', 15))
        ax.set_title(f'{ds_name}\n{best_name} — Feature Importance', fontweight='bold')
        ax.set_xlabel('Importance')
    else:
        ax.text(0.5, 0.5, 'Feature importance\nnot available', ha='center', va='center')
        ax.set_title(f'{ds_name} — {best_name}', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print('💾 Saved: plots/feature_importance.png')

## Final Model Selection

### Selection Criteria (prioritised order)
1. **Recall** — minimise false negatives (missed disease cases are costly)
2. **F1-Score** — balance precision and recall
3. **ROC-AUC** — overall discriminative ability
4. **Balanced Accuracy** — robust to class imbalance
5. **MCC** — most informative single metric for imbalanced classification

> ⚠️ Accuracy alone is misleading for imbalanced healthcare data. A model predicting all-negative achieves 91% accuracy on Heart Disease data but has 0% Recall.

In [ ]:
# ── 13. Final Model Selection & Summary ───────────────────────────────────

print('='*70)
print('FINAL MODEL EVALUATION SUMMARY')
print('='*70)

priority_cols = ['Dataset', 'Model', 'Recall', 'F1', 'ROC-AUC', 'Balanced Accuracy', 'MCC', 'Accuracy']
display(metrics_df[priority_cols].round(4))

print('\n🏆 RECOMMENDED MODELS:')
for ds in ['Cardiovascular', 'Diabetes', 'Heart Disease']:
    ds_metrics = metrics_df[metrics_df['Dataset'] == ds].sort_values('Recall', ascending=False).iloc[0]
    print(f'\n  {ds}:')
    print(f'    Best Model       : {ds_metrics["Model"]}')
    print(f'    Recall           : {ds_metrics["Recall"]:.4f}')
    print(f'    F1               : {ds_metrics["F1"]:.4f}')
    print(f'    ROC-AUC          : {ds_metrics["ROC-AUC"]:.4f}')
    print(f'    Balanced Accuracy: {ds_metrics["Balanced Accuracy"]:.4f}')
    print(f'    MCC              : {ds_metrics["MCC"]:.4f}')

In [ ]:
# ── 14. Save Best Models and Artifacts ────────────────────────────────────

metrics_df.to_csv('artifacts/metrics.csv', index=False)
print('💾 Saved: artifacts/metrics.csv')

for ds_name, m_name in best_models_per_ds.items():
    model = tuned_models[ds_name][m_name]
    safe_ds = ds_name.lower().replace(' ', '_')
    fname = f'artifacts/best_model_{safe_ds}.joblib'
    joblib.dump(model, fname)
    print(f'💾 Saved: {fname}  ({m_name})')

print('\n📁 Final artifact directory:')
for f in sorted(os.listdir('artifacts')):
    size_kb = os.path.getsize(f'artifacts/{f}') / 1024
    print(f'   artifacts/{f}  ({size_kb:.1f} KB)')

In [ ]:
# ── 15. Classification Reports ────────────────────────────────────────────
for ds_name, m_name in best_models_per_ds.items():
    res    = test_results[ds_name]['models'][m_name]
    y_test = test_results[ds_name]['y_test']
    print(f'\n{'='*55}')
    print(f'Classification Report — {ds_name} | {m_name}')
    print('='*55)
    print(classification_report(y_test, res['y_pred'],
                                target_names=['No Disease', 'Disease']))

## Conclusion

### Summary of Results

| Dataset | Best Model | Recall | F1 | ROC-AUC | Key Challenge |
|---------|-----------|--------|-----|---------|---------------|
| Cardiovascular | (see output) | — | — | — | Outlier BP values |
| Diabetes | (see output) | — | — | — | 14% minority class |
| Heart Disease | (see output) | — | — | — | 9% minority class |

### Why We Prioritised Recall
In disease prediction, a **false negative** (missed diagnosis) carries a far higher cost than a **false positive** (unnecessary follow-up). Our evaluation framework, resampling strategy, and hyperparameter tuning scoring (`scoring='recall'`) all reflect this clinical priority.

### Recommended Next Steps
1. **Threshold tuning**: Lower the decision threshold (default 0.5) to further boost Recall at acceptable Precision cost.
2. **Calibration**: Apply `CalibratedClassifierCV` to ensure probability outputs are well-calibrated for clinical use.
3. **Explainability**: Add SHAP analysis for individual prediction explanations — critical for clinician trust.
4. **External validation**: Test on held-out temporal cohort to assess generalisation.
5. **Model deployment**: Wrap best models in a FastAPI service with the saved `preprocessor.joblib` for real-time inference.